# Download do Dataset LUNA16

Antes de comecar qualquer analise, a gente precisa dos dados. O dataset LUNA16 esta hospedado no Zenodo em duas partes:

- **Part 1** (https://zenodo.org/records/3723295): `annotations.csv`, `candidates.csv` e subset0 a subset6
- **Part 2** (https://zenodo.org/records/4121926): subset7 a subset9

Cada subset contem CT scans no formato `.mhd` + `.raw` (pares de arquivos). O download total e de ~67 GB, entao se prepare.

O codigo abaixo suporta **resume** em caso de queda de conexao. Arquivos ja existentes sao automaticamente ignorados.

In [ ]:
import os
import zipfile
from pathlib import Path
import shutil

import requests
from tqdm import tqdm

## Configuracao

Vamos definir os caminhos e as URLs de cada arquivo. A estrutura final fica em `data/luna/` com os CSVs e os subsets.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LUNA_DIR = PROJECT_ROOT / "data" / "luna"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LUNA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projeto: {PROJECT_ROOT}")
print(f"Downloads: {RAW_DIR}")
print(f"Dataset: {LUNA_DIR}")

In [ ]:
ZENODO_PART1 = "https://zenodo.org/records/3723295/files"
ZENODO_PART2 = "https://zenodo.org/records/4121926/files"

LUNA16_FILES = {
    "annotations.csv": f"{ZENODO_PART1}/annotations.csv?download=1",
    "candidates.csv": f"{ZENODO_PART1}/candidates.csv?download=1",
    "subset0.zip": f"{ZENODO_PART1}/subset0.zip?download=1",
    "subset1.zip": f"{ZENODO_PART1}/subset1.zip?download=1",
    "subset2.zip": f"{ZENODO_PART1}/subset2.zip?download=1",
    "subset3.zip": f"{ZENODO_PART1}/subset3.zip?download=1",
    "subset4.zip": f"{ZENODO_PART1}/subset4.zip?download=1",
    "subset5.zip": f"{ZENODO_PART1}/subset5.zip?download=1",
    "subset6.zip": f"{ZENODO_PART1}/subset6.zip?download=1",
    "subset7.zip": f"{ZENODO_PART2}/subset7.zip?download=1",
    "subset8.zip": f"{ZENODO_PART2}/subset8.zip?download=1",
    "subset9.zip": f"{ZENODO_PART2}/subset9.zip?download=1",
}

print(f"Total de arquivos: {len(LUNA16_FILES)}")

## Funcoes de download

As funcoes abaixo cuidam de baixar e extrair os arquivos. O download suporta resume (se cair a conexao, ele continua de onde parou) e pula arquivos que ja existem.

In [ ]:
def download_file(url, dest_path, chunk_size=8192):
    """Baixa um arquivo com suporte a resume."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    head = requests.head(url, allow_redirects=True, timeout=30)
    remote_size = int(head.headers.get("Content-Length", 0))

    if dest_path.exists():
        local_size = dest_path.stat().st_size
        if local_size >= remote_size and remote_size > 0:
            print(f"  [OK] {dest_path.name} ja existe ({local_size / 1e6:.1f} MB)")
            return False
    else:
        local_size = 0

    headers = {}
    if local_size > 0:
        headers["Range"] = f"bytes={local_size}-"
        print(f"  Resumindo {dest_path.name} a partir de {local_size / 1e6:.1f} MB...")

    response = requests.get(url, headers=headers, stream=True, timeout=60)
    response.raise_for_status()

    content_length = int(response.headers.get("Content-Length", 0))
    total = content_length if content_length > 0 else (remote_size - local_size)
    mode = "ab" if local_size > 0 else "wb"

    with open(dest_path, mode) as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc=dest_path.name) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return True

In [ ]:
def extract_subset_zip(zip_path, luna_dir):
    """Extrai um subset zip para a estrutura correta."""
    zip_path = Path(zip_path)
    luna_dir = Path(luna_dir)
    subset_name = zip_path.stem
    target_dir = luna_dir / subset_name

    if target_dir.exists() and any(target_dir.glob("*.mhd")):
        n = len(list(target_dir.glob("*.mhd")))
        print(f"  [OK] {subset_name} ja extraido ({n} scans)")
        return

    print(f"  Extraindo {zip_path.name}...")
    temp_dir = luna_dir / f"_temp_{subset_name}"

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_dir)

    # O zip cria pasta aninhada (subset0/subset0/) -- mover pro nivel certo
    nested_dir = temp_dir / subset_name / subset_name
    if nested_dir.exists():
        source = nested_dir
    elif (temp_dir / subset_name).exists():
        source = temp_dir / subset_name
    else:
        source = temp_dir

    target_dir.mkdir(parents=True, exist_ok=True)
    for f in source.iterdir():
        shutil.move(str(f), str(target_dir / f.name))

    shutil.rmtree(temp_dir, ignore_errors=True)
    mhd_count = len(list(target_dir.glob("*.mhd")))
    print(f"  [OK] {subset_name} extraido ({mhd_count} scans)")

In [ ]:
def download_and_extract(file_name, url, raw_dir, luna_dir):
    """Baixa e extrai um arquivo do LUNA16."""
    raw_dir, luna_dir = Path(raw_dir), Path(luna_dir)

    if file_name.endswith(".csv"):
        dest = luna_dir / file_name
        if dest.exists() and dest.stat().st_size > 0:
            print(f"  [OK] {file_name} ja existe em luna/")
            return
        download_file(url, dest)
        return

    if file_name.endswith(".zip"):
        subset_name = file_name.replace(".zip", "")
        target_dir = luna_dir / subset_name
        if target_dir.exists() and any(target_dir.glob("*.mhd")):
            print(f"  [OK] {subset_name} ja existe -- pulando download")
            return
        zip_path = raw_dir / file_name
        download_file(url, zip_path)
        extract_subset_zip(zip_path, luna_dir)

## Teste com um subset

Antes de baixar o dataset inteiro (~67 GB), vamos testar com um unico subset pra garantir que tudo funciona:

In [ ]:
test_files = ["annotations.csv", "candidates.csv", "subset0.zip"]

for file_name in test_files:
    print(f"\nProcessando: {file_name}")
    url = LUNA16_FILES[file_name]
    download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)

## Verificacao rapida

Vamos ver o que temos em `data/luna/` ate agora:

In [ ]:
for item in sorted(LUNA_DIR.iterdir()):
    if item.name.startswith("."):
        continue
    if item.is_dir():
        mhd_files = list(item.glob("*.mhd"))
        print(f"  {item.name}/  ({len(mhd_files)} scans)")
    else:
        print(f"  {item.name}  ({item.stat().st_size / 1e6:.1f} MB)")

## Download completo

Se o teste acima funcionou, rode a celula abaixo pra baixar todos os 10 subsets. Arquivos ja presentes sao ignorados automaticamente.

In [ ]:
for file_name, url in LUNA16_FILES.items():
    print(f"\nProcessando: {file_name}")
    try:
        download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)
    except Exception as e:
        print(f"  [ERRO] {file_name}: {e}")
        print(f"  Re-execute esta celula para tentar novamente.")

## Verificacao final

In [ ]:
for csv_name in ["annotations.csv", "candidates.csv"]:
    csv_path = LUNA_DIR / csv_name
    status = "presente" if csv_path.exists() else "FALTANDO"
    print(f"  {csv_name}: {status}")

print()
total_scans = 0
for i in range(10):
    subset_dir = LUNA_DIR / f"subset{i}"
    if subset_dir.exists():
        n = len(list(subset_dir.glob("*.mhd")))
        total_scans += n
        print(f"  subset{i}: {n} scans")
    else:
        print(f"  subset{i}: FALTANDO")

print(f"\nTotal de CT scans: {total_scans}")